# LeetCode 1500 — Design a File Sharing System bằng Python

Link đề: https://leetcode.com/problems/design-a-file-sharing-system/ · Độ khó: **Medium** · Chủ đề: Hash Table, Design, Sorting, Heap (Priority Queue), Data Stream

> **Ghi chú Premium:** đây là bài Premium 🔒. Đề bài và chữ ký trong notebook được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1500/). Nếu có tài khoản Premium, bạn nên đối chiếu lại đề gốc.

Skeleton LeetCode (theo lời giải Python của doocs):

```python
class FileSharing:
    def __init__(self, m: int):
        ...

    def join(self, ownedChunks: list[int]) -> int:
        ...

    def leave(self, userID: int) -> None:
        ...

    def request(self, userID: int, chunkID: int) -> list[int]:
        ...

# Your FileSharing object will be instantiated and called as such:
# obj = FileSharing(m)
# param_1 = obj.join(ownedChunks)
# obj.leave(userID)
# param_3 = obj.request(userID,chunkID)
```

## 1. Mục tiêu học

1. Thiết kế lớp có **nhiều thao tác** và chọn cấu trúc dữ liệu cho từng thao tác: `dict` người dùng → tập mảnh, `set`, min-heap.
2. Bài toán con kinh điển: **cấp ID nguyên dương nhỏ nhất chưa dùng**, có **tái sử dụng** ID khi người dùng rời đi ⇒ bộ đếm + **min-heap** các ID đã giải phóng.
3. Xây **chỉ mục ngược** mảnh → tập người dùng để trả lời `request` nhanh, và giữ hai chỉ mục **đồng bộ** khi `join/leave/request`.
4. Tự dựng **bộ chạy lệnh giả lập** kiểu LeetCode (`["FileSharing","join",...]` + tham số) để test, và đối chiếu nhiều cài đặt trên chuỗi lệnh ngẫu nhiên.
5. Thảo luận các câu hỏi follow-up (định danh theo IP, join/leave liên tục, nhiều file).

## 2. Đề bài & ví dụ

Một file rất lớn gồm `m` mảnh (chunk) đánh số `1..m`, được chia sẻ trong một hệ thống.

- Khi người dùng **tham gia**, hệ thống cấp cho họ một **ID duy nhất**: số nguyên dương **nhỏ nhất** chưa bị người dùng nào khác (đang trong hệ thống) chiếm. Khi người dùng rời đi, ID của họ được phép **dùng lại**.
- Người dùng có thể **yêu cầu** một mảnh; hệ thống trả về danh sách ID của **mọi** người dùng đang sở hữu mảnh đó, **sắp xếp tăng dần**. Nếu danh sách khác rỗng, người yêu cầu **nhận được** mảnh đó (từ đó cũng sở hữu nó).

Cài đặt lớp `FileSharing`:
- `FileSharing(m)`: khởi tạo với file `m` mảnh.
- `join(ownedChunks) -> int`: người dùng mới tham gia, sở hữu sẵn các mảnh `ownedChunks`; trả về ID được cấp.
- `leave(userID) -> None`: người dùng rời hệ thống; không thể lấy mảnh từ họ nữa.
- `request(userID, chunkID) -> list[int]`: trả về danh sách ID (tăng dần) đang sở hữu `chunkID`.

**Ví dụ:**
```
["FileSharing","join","join","join","request","request","leave","request","leave","join"]
[[4],[[1,2]],[[2,3]],[[4]],[1,3],[2,2],[1],[2,1],[2],[[]]]
Output: [null,1,2,3,[2],[1,2],null,[],null,1]
```
- `join([1,2])` → 1; `join([2,3])` → 2; `join([4])` → 3.
- `request(1, 3)` → `[2]` (chỉ user 2 có mảnh 3); user 1 giờ có `[1,2,3]`.
- `request(2, 2)` → `[1,2]`.
- `leave(1)`; `request(2, 1)` → `[]` (không ai còn mảnh 1).
- `leave(2)`; `join([])` → **1** (ID 1 và 2 đã trống, cấp nhỏ nhất).

**Ràng buộc:** `1 <= m <= 10^5`; `0 <= len(ownedChunks) <= min(100, m)`; `1 <= ownedChunks[i] <= m`, các giá trị phân biệt; `1 <= chunkID <= m`; `userID` luôn là người dùng đang trong hệ thống nếu cấp ID đúng; tổng số lời gọi `join`, `leave`, `request` tối đa `10^4`; mỗi `leave` có `join` tương ứng.

In [1]:
import heapq
import random
import time
from collections import defaultdict
from typing import List

EX_CMDS = ["FileSharing", "join", "join", "join", "request", "request", "leave", "request", "leave", "join"]
EX_ARGS = [[4], [[1, 2]], [[2, 3]], [[4]], [1, 3], [2, 2], [1], [2, 1], [2], [[]]]
EX_OUT = [None, 1, 2, 3, [2], [1, 2], None, [], None, 1]
print(list(zip(EX_CMDS, EX_ARGS, EX_OUT)))

[('FileSharing', [4], None), ('join', [[1, 2]], 1), ('join', [[2, 3]], 2), ('join', [[4]], 3), ('request', [1, 3], [2]), ('request', [2, 2], [1, 2]), ('leave', [1], None), ('request', [2, 1], []), ('leave', [2], None), ('join', [[]], 1)]


**Bộ chạy lệnh giả lập** (bản tự dựng, mô phỏng cách LeetCode gọi lớp thiết kế): lệnh đầu tạo đối tượng, các lệnh sau gọi method cùng tên với tham số tương ứng, gom kết quả (`None` cho method không trả về).

In [2]:
def run(cls, cmds, args):
    obj = None
    out = []
    for c, a in zip(cmds, args):
        if obj is None:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, c)(*a))
    return out

print("bộ chạy lệnh đã sẵn sàng")

bộ chạy lệnh đã sẵn sàng


## 3. Kiến thức nền

### 3.1 `dict` của `set`

`user_chunks[uid] = set(...)` lưu các mảnh của từng người. `chunk in s` là `O(1)` trung bình. `defaultdict(set)` tự tạo tập rỗng khi truy cập khoá mới.

In [3]:
user_chunks = defaultdict(set)
user_chunks[1] = {1, 2}
user_chunks[2].add(3)          # tự tạo set rỗng rồi thêm
print(dict(user_chunks), 2 in user_chunks[1], 3 in user_chunks[1])

{1: {1, 2}, 2: {3}} True False


### 3.2 Min-heap `heapq` để lấy số nhỏ nhất

`heappush` / `heappop` đều `O(log k)`; `heappop` luôn trả phần tử **nhỏ nhất**. Dùng để giữ các ID đã được giải phóng.

In [4]:
freed = []
for uid in [5, 2, 7, 3]:
    heapq.heappush(freed, uid)
print("lấy ra lần lượt:", [heapq.heappop(freed) for _ in range(4)])

lấy ra lần lượt: [2, 3, 5, 7]


### 3.3 `sorted()` và thứ tự duyệt của `set`/`dict`

`dict` duyệt theo **thứ tự chèn**; `set` duyệt theo thứ tự nội bộ của bảng băm — **không** đảm bảo tăng dần. Đề yêu cầu kết quả tăng dần ⇒ luôn `sorted(...)`.

In [5]:
d = {}
d[2] = "user 2"; d[1] = "user 1"       # 1 được chèn sau
s = set(); s.add(9); s.add(1)
print("dict:", list(d), "| set:", list(s), "| sorted:", sorted(s))

dict: [2, 1] | set: [9, 1] | sorted: [1, 9]


### 3.4 Lớp (class) và trạng thái

Mỗi method đọc/sửa các thuộc tính `self.xxx`. Thiết kế tốt = chọn thuộc tính sao cho mọi method đều nhanh, và **mọi thuộc tính luôn nhất quán** với nhau sau mỗi lời gọi.

In [6]:
class Counter2:
    def __init__(self):
        self.value = 0
    def inc(self):
        self.value += 1
        return self.value

c = Counter2()
print(c.inc(), c.inc(), c.value)

1 2 2


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay ví dụ với hai bảng: `users` (ID → tập mảnh) và "ID trống":

| lệnh | users sau lệnh | ID trống | trả về |
|---|---|---|---|
| FileSharing(4) | {} | {1,2,3,…} | null |
| join([1,2]) | {1:{1,2}} | {2,3,…} | 1 |
| join([2,3]) | {1:{1,2}, 2:{2,3}} | {3,…} | 2 |
| join([4]) | + {3:{4}} | {4,…} | 3 |
| request(1,3) | 1 nhận mảnh 3 ⇒ {1:{1,2,3}} | | [2] |
| request(2,2) | không đổi (2 đã có) | | [1,2] |
| leave(1) | {2:{2,3}, 3:{4}} | {1,4,…} | null |
| request(2,1) | không ai có ⇒ không nhận | | [] |
| leave(2) | {3:{4}} | {1,2,4,…} | null |
| join([]) | + {1:{}} | {2,4,…} | **1** |

Hai câu hỏi thiết kế:
1. "ID trống nhỏ nhất" — tập vô hạn `{1,2,3,…}` biểu diễn thế nào? ⇒ **bộ đếm** `cur` (mọi ID `> cur` chưa từng dùng) + **heap** chứa các ID `≤ cur` đã được trả lại.
2. `request` — duyệt mọi người dùng (đơn giản) hay giữ **chỉ mục ngược** mảnh → người dùng (nhanh hơn)?

## 5. Cách 1 — Brute Force / baseline: quét tìm ID nhỏ nhất, quét mọi người dùng

**Ý tưởng:**
- `join`: thử `uid = 1, 2, 3, …` tới khi gặp ID chưa có trong `users`.
- `leave`: xoá khỏi `users`.
- `request`: duyệt mọi người dùng, gom người có mảnh, sắp xếp; nếu khác rỗng thì người yêu cầu nhận mảnh.

In [7]:
class FileSharingBrute:
    def __init__(self, m: int):
        self.m = m
        self.users = {}                       # uid -> set mảnh

    def join(self, ownedChunks: list[int]) -> int:
        uid = 1
        while uid in self.users:              # quét O(số người dùng)
            uid += 1
        self.users[uid] = set(ownedChunks)
        return uid

    def leave(self, userID: int) -> None:
        del self.users[userID]

    def request(self, userID: int, chunkID: int) -> list[int]:
        res = sorted(u for u, ch in self.users.items() if chunkID in ch)
        if res:
            self.users[userID].add(chunkID)
        return res

out = run(FileSharingBrute, EX_CMDS, EX_ARGS)
print(out, out == EX_OUT)

[None, 1, 2, 3, [2], [1, 2], None, [], None, 1] True


**Độ phức tạp:** gọi `U` = số người dùng đang online.
- `join`: `O(U + c)` (`c = len(ownedChunks)`); `leave`: `O(1)`; `request`: `O(U log U)`.

**Nút thắt:** `join` quét từ 1 mỗi lần. Với chuỗi "join 10^4 lần liên tiếp" ⇒ `~5·10^7` phép kiểm tra. `request` cũng duyệt **mọi** người dùng, kể cả khi chỉ vài người có mảnh đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — ID nhỏ nhất chưa dùng.** Chia ID thành hai loại:
- ID **chưa từng cấp**: đúng là các số `> cur` (bộ đếm tăng dần). Nhỏ nhất là `cur + 1`.
- ID **đã cấp rồi được trả lại**: luôn `≤ cur`, lưu trong **min-heap** `reused`.

Mọi ID trong heap đều `< cur + 1` ⇒ **nếu heap khác rỗng thì lấy đỉnh heap, ngược lại lấy `cur + 1`**. Invariant: `{ID trống} = reused ∪ {cur+1, cur+2, …}`.

**Insight 2 — Chỉ mục ngược.** Giữ thêm `owners[chunk] = set(ID đang có mảnh đó)`. Khi đó `request` chỉ cần `sorted(owners[chunkID])` — chi phí theo **số người có mảnh**, không theo tổng số người dùng. Cái giá: `join/leave` phải cập nhật `owners` cho từng mảnh của người đó (≤ 100 mảnh lúc join, nhưng có thể nhiều hơn sau các lần `request`).

**Quy tắc đồng bộ:**
- `join(chunks)`: cấp `uid`; `user_chunks[uid] = set(chunks)`; với mỗi `c`: `owners[c].add(uid)`.
- `leave(uid)`: với mỗi `c` trong `user_chunks[uid]`: `owners[c].discard(uid)`; xoá `user_chunks[uid]`; `heappush(reused, uid)`.
- `request(uid, c)`: `res = sorted(owners[c])`; nếu `res` khác rỗng: `owners[c].add(uid)`, `user_chunks[uid].add(c)`.

## 7. Các cách tiếp theo

### Cách 2 — Bộ đếm + heap tái sử dụng ID (cách của doocs)

`join`/`leave` nhanh nhờ heap; `request` vẫn duyệt mọi người dùng (chấp nhận được vì tổng số lời gọi ≤ `10^4`). Doocs còn trả `[]` nếu `chunkID` ngoài `[1, m]` (phòng thủ, theo ràng buộc thì không xảy ra).

In [8]:
class FileSharingHeap:
    def __init__(self, m: int):
        self.cur = 0
        self.chunks = m
        self.reused = []
        self.user_chunks = defaultdict(set)

    def join(self, ownedChunks: list[int]) -> int:
        if self.reused:
            userID = heapq.heappop(self.reused)
        else:
            self.cur += 1
            userID = self.cur
        self.user_chunks[userID] = set(ownedChunks)
        return userID

    def leave(self, userID: int) -> None:
        heapq.heappush(self.reused, userID)
        self.user_chunks.pop(userID)

    def request(self, userID: int, chunkID: int) -> list[int]:
        if chunkID < 1 or chunkID > self.chunks:
            return []
        res = []
        for k, v in self.user_chunks.items():
            if chunkID in v:
                res.append(k)
        if res:
            self.user_chunks[userID].add(chunkID)
        return sorted(res)

out = run(FileSharingHeap, EX_CMDS, EX_ARGS)
print(out, out == EX_OUT)

[None, 1, 2, 3, [2], [1, 2], None, [], None, 1] True


**Độ phức tạp:** `join`: `O(log U + c)`; `leave`: `O(log U)`; `request`: `O(U log U)` (duyệt + sắp xếp). Bộ nhớ `O(U + tổng số mảnh đang được giữ)`.

Lưu ý: `sorted(res)` là **bắt buộc** — `user_chunks` duyệt theo thứ tự chèn, mà một ID nhỏ được cấp lại sẽ bị chèn **sau** các ID lớn (xem Lỗi 4).

### Cách 3 — Heap + chỉ mục ngược mảnh → người dùng

`request` không còn duyệt toàn bộ người dùng.

In [9]:
class FileSharingIndex:
    def __init__(self, m: int):
        self.m = m
        self.cur = 0
        self.reused = []                       # min-heap ID đã trả lại
        self.user_chunks = {}                  # uid -> set mảnh
        self.owners = defaultdict(set)         # mảnh -> set uid

    def join(self, ownedChunks: list[int]) -> int:
        if self.reused:
            uid = heapq.heappop(self.reused)
        else:
            self.cur += 1
            uid = self.cur
        self.user_chunks[uid] = set(ownedChunks)
        for c in ownedChunks:
            self.owners[c].add(uid)
        return uid

    def leave(self, userID: int) -> None:
        for c in self.user_chunks.pop(userID):
            self.owners[c].discard(userID)
        heapq.heappush(self.reused, userID)

    def request(self, userID: int, chunkID: int) -> list[int]:
        res = sorted(self.owners[chunkID])
        if res:
            self.owners[chunkID].add(userID)
            self.user_chunks[userID].add(chunkID)
        return res

out = run(FileSharingIndex, EX_CMDS, EX_ARGS)
print(out, out == EX_OUT)

[None, 1, 2, 3, [2], [1, 2], None, [], None, 1] True


**Độ phức tạp:** gọi `k` = số người đang có mảnh được hỏi, `c_u` = số mảnh của người dùng `u`.
- `join`: `O(log U + c)`; `leave`: `O(log U + c_u)`; `request`: `O(k log k)`.

Đây là lựa chọn tốt khi `request` chiếm đa số lời gọi và mỗi mảnh chỉ có ít người giữ. Nếu người dùng gom được rất nhiều mảnh (hàng chục nghìn) rồi `leave`, chi phí `leave` lớn — đó chính là câu follow-up thứ ba của đề.

### Cách 4 — Biến thể: thay heap bằng `set` ID trống + `min()`

Để thấy vai trò của heap: giữ `free` là `set` các ID đã trả lại, `join` lấy `min(free)` — đúng nhưng `O(|free|)` mỗi lần. Hữu ích như một cài đặt tham chiếu thứ hai.

In [10]:
class FileSharingMinSet(FileSharingIndex):
    def __init__(self, m: int):
        super().__init__(m)
        self.free = set()

    def join(self, ownedChunks: list[int]) -> int:
        if self.free:
            uid = min(self.free)               # O(|free|)
            self.free.remove(uid)
        else:
            self.cur += 1
            uid = self.cur
        self.user_chunks[uid] = set(ownedChunks)
        for c in ownedChunks:
            self.owners[c].add(uid)
        return uid

    def leave(self, userID: int) -> None:
        for c in self.user_chunks.pop(userID):
            self.owners[c].discard(userID)
        self.free.add(userID)

out = run(FileSharingMinSet, EX_CMDS, EX_ARGS)
print(out, out == EX_OUT)

[None, 1, 2, 3, [2], [1, 2], None, [], None, 1] True


## 8. TRACE từng bước

Chạy ví dụ đề trên Cách 3, in toàn bộ trạng thái (`cur`, heap `reused`, `user_chunks`, `owners`) sau mỗi lệnh.

In [11]:
def trace_index(cmds, args):
    obj = None
    for c, a in zip(cmds, args):
        if obj is None:
            obj = FileSharingIndex(*a); res = None
        else:
            res = getattr(obj, c)(*a)
        owners = {k: sorted(v) for k, v in sorted(obj.owners.items()) if v}
        users = {k: sorted(v) for k, v in sorted(obj.user_chunks.items())}
        print(f"{c:11} {str(a):10} -> {str(res):7} | cur={obj.cur} reused={sorted(obj.reused)} users={users} owners={owners}")

trace_index(EX_CMDS, EX_ARGS)

FileSharing [4]        -> None    | cur=0 reused=[] users={} owners={}
join        [[1, 2]]   -> 1       | cur=1 reused=[] users={1: [1, 2]} owners={1: [1], 2: [1]}
join        [[2, 3]]   -> 2       | cur=2 reused=[] users={1: [1, 2], 2: [2, 3]} owners={1: [1], 2: [1, 2], 3: [2]}
join        [[4]]      -> 3       | cur=3 reused=[] users={1: [1, 2], 2: [2, 3], 3: [4]} owners={1: [1], 2: [1, 2], 3: [2], 4: [3]}
request     [1, 3]     -> [2]     | cur=3 reused=[] users={1: [1, 2, 3], 2: [2, 3], 3: [4]} owners={1: [1], 2: [1, 2], 3: [1, 2], 4: [3]}
request     [2, 2]     -> [1, 2]  | cur=3 reused=[] users={1: [1, 2, 3], 2: [2, 3], 3: [4]} owners={1: [1], 2: [1, 2], 3: [1, 2], 4: [3]}
leave       [1]        -> None    | cur=3 reused=[1] users={2: [2, 3], 3: [4]} owners={2: [2], 3: [2], 4: [3]}
request     [2, 1]     -> []      | cur=3 reused=[1] users={2: [2, 3], 3: [4]} owners={2: [2], 3: [2], 4: [3]}
leave       [2]        -> None    | cur=3 reused=[1, 2] users={3: [4]} owners={4: [3]}
jo

Trace thêm một chuỗi nhấn mạnh việc **tái sử dụng ID nhỏ nhất**: ba người vào, người 2 rồi người 1 rời đi, sau đó hai người mới vào — phải nhận lần lượt ID 1 rồi 2 (không phải 4, 5), và người thứ ba mới nhận 4:

In [12]:
cmds2 = ["FileSharing", "join", "join", "join", "leave", "leave", "join", "join", "join", "request"]
args2 = [[5], [[1]], [[2]], [[3]], [2], [1], [[5]], [[]], [[1, 5]], [3, 5]]
trace_index(cmds2, args2)
print("kết quả:", run(FileSharingIndex, cmds2, args2))

FileSharing [5]        -> None    | cur=0 reused=[] users={} owners={}
join        [[1]]      -> 1       | cur=1 reused=[] users={1: [1]} owners={1: [1]}
join        [[2]]      -> 2       | cur=2 reused=[] users={1: [1], 2: [2]} owners={1: [1], 2: [2]}
join        [[3]]      -> 3       | cur=3 reused=[] users={1: [1], 2: [2], 3: [3]} owners={1: [1], 2: [2], 3: [3]}
leave       [2]        -> None    | cur=3 reused=[2] users={1: [1], 3: [3]} owners={1: [1], 3: [3]}
leave       [1]        -> None    | cur=3 reused=[1, 2] users={3: [3]} owners={3: [3]}
join        [[5]]      -> 1       | cur=3 reused=[2] users={1: [5], 3: [3]} owners={3: [3], 5: [1]}
join        [[]]       -> 2       | cur=3 reused=[] users={1: [5], 2: [], 3: [3]} owners={3: [3], 5: [1]}
join        [[1, 5]]   -> 4       | cur=4 reused=[] users={1: [5], 2: [], 3: [3], 4: [1, 5]} owners={1: [4], 3: [3], 5: [1, 4]}
request     [3, 5]     -> [1, 4]  | cur=4 reused=[] users={1: [5], 2: [], 3: [3, 5], 4: [1, 5]} owners={1: [4],

Bảng tóm tắt heap ở chuỗi trên:

| lệnh | heap `reused` | `cur` | ID cấp |
|---|---|---|---|
| join ×3 | [] | 3 | 1, 2, 3 |
| leave(2) | [2] | 3 | |
| leave(1) | [1, 2] | 3 | |
| join | [2] | 3 | **1** (đỉnh heap) |
| join | [] | 3 | **2** |
| join | [] | 4 | **4** (heap rỗng ⇒ `cur + 1`) |

## 9. Code nộp LeetCode

Bản chính: Cách 3 (heap tái sử dụng ID + chỉ mục ngược). Trên LeetCode cần `import heapq` và `defaultdict` (hoặc dùng `from heapq import heappush, heappop` như doocs).

In [13]:
import heapq
from collections import defaultdict

class FileSharing:
    def __init__(self, m: int):
        self.m = m
        self.cur = 0                           # mọi ID > cur chưa từng được cấp
        self.reused = []                       # min-heap ID đã được trả lại (đều <= cur)
        self.user_chunks = {}                  # uid -> set mảnh
        self.owners = defaultdict(set)         # mảnh -> set uid đang giữ

    def join(self, ownedChunks: list[int]) -> int:
        if self.reused:
            uid = heapq.heappop(self.reused)
        else:
            self.cur += 1
            uid = self.cur
        self.user_chunks[uid] = set(ownedChunks)
        for c in ownedChunks:
            self.owners[c].add(uid)
        return uid

    def leave(self, userID: int) -> None:
        for c in self.user_chunks.pop(userID):
            self.owners[c].discard(userID)
        heapq.heappush(self.reused, userID)

    def request(self, userID: int, chunkID: int) -> list[int]:
        res = sorted(self.owners[chunkID])
        if res:                                # nhận mảnh chỉ khi có người giữ
            self.owners[chunkID].add(userID)
            self.user_chunks[userID].add(chunkID)
        return res

assert run(FileSharing, EX_CMDS, EX_ARGS) == EX_OUT
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**ID nhỏ nhất (invariant I1):** sau mỗi lời gọi, tập ID **đang trống** đúng bằng `R ∪ {cur+1, cur+2, …}` với `R` = các phần tử của heap, và mọi phần tử của `R` đều `≤ cur`.
- Ban đầu `R = ∅`, `cur = 0` ⇒ trống là `{1, 2, …}` ✓.
- `join`: nếu `R ≠ ∅`, đỉnh heap là phần tử nhỏ nhất của `R`, và nó `≤ cur < cur + 1` ⇒ là ID trống nhỏ nhất. Nếu `R = ∅`, nhỏ nhất là `cur + 1`; tăng `cur` giữ I1.
- `leave(u)`: `u` đang được dùng nên `u ≤ cur`; đẩy `u` vào heap ⇒ I1 vẫn đúng.

**Nhất quán hai chỉ mục (invariant I2):** `u ∈ owners[c]` ⇔ `u` đang online và `c ∈ user_chunks[u]`. Mỗi method cập nhật **cả hai phía** cùng lúc: `join` thêm cả hai, `leave` xoá `u` khỏi mọi `owners[c]` với `c` thuộc tập mảnh của `u` (đúng các chỗ `u` có mặt theo I2), `request` thêm cả hai khi `res` khác rỗng.

Từ I2: `owners[chunkID]` đúng là tập người đang giữ mảnh ⇒ `sorted(...)` là đáp án. Người yêu cầu nhận mảnh chỉ khi danh sách khác rỗng — đúng đặc tả.

## 11. Phân tích Time / Space

Gọi `U` = số người online, `c` = `len(ownedChunks)` (≤ 100), `c_u` = số mảnh người `u` đang giữ, `k` = số người giữ mảnh được hỏi, `Q ≤ 10^4` = số lời gọi.

| Cách | join | leave | request | Space |
|---|---|---|---|---|
| 1. Quét | `O(U + c)` | `O(1)` | `O(U log U)` | `O(U + S)` |
| 2. Heap (doocs) | `O(log U + c)` | `O(log U)` | `O(U log U)` | `O(U + S)` |
| 3. Heap + chỉ mục ngược | `O(log U + c)` | `O(log U + c_u)` | `O(k log k)` | `O(U + 2S)` |
| 4. Set + `min` | `O(|free| + c)` | `O(c_u)` | `O(k log k)` | `O(U + 2S)` |

`S` = tổng số cặp (người, mảnh) đang tồn tại. Với `Q ≤ 10^4` mọi cách đều qua; Cách 3 tốt nhất khi `request` nhiều và `U` lớn.

## 12. Bộ test

### 12.1 Ví dụ đề + các chuỗi lệnh biên (bản giả lập)

In [14]:
IMPLS = {
    "brute": FileSharingBrute,
    "heap(doocs)": FileSharingHeap,
    "index": FileSharingIndex,
    "minset": FileSharingMinSet,
    "submit": FileSharing,
}
cases = [
    (EX_CMDS, EX_ARGS, EX_OUT),
    (cmds2, args2, [None, 1, 2, 3, None, None, 1, 2, 4, [1, 4]]),
    (["FileSharing", "join", "request"], [[1], [[]], [1, 1]], [None, 1, []]),          # không ai có mảnh
    (["FileSharing", "join", "request"], [[1], [[1]], [1, 1]], [None, 1, [1]]),        # tự hỏi mảnh mình có
    (["FileSharing", "join", "leave", "join", "leave", "join"], [[3], [[1]], [1], [[2]], [1], [[3]]],
     [None, 1, None, 1, None, 1]),                                                    # vào ra liên tục
    (["FileSharing", "join", "join", "request", "leave", "request"],
     [[2], [[2]], [[]], [2, 2], [1], [2, 2]], [None, 1, 2, [1], None, [2]]),           # nhận mảnh rồi người gốc rời đi
]
for name, cls in IMPLS.items():
    for cmds, args, want in cases:
        got = run(cls, cmds, args)
        assert got == want, (name, cmds, got, want)
print("Mọi cài đặt đúng với", len(cases), "chuỗi lệnh")

Mọi cài đặt đúng với 6 chuỗi lệnh


### 12.2 Chuỗi lệnh ngẫu nhiên, đối chiếu giữa các cài đặt (seed cố định)

Sinh chuỗi hợp lệ: `leave`/`request` chỉ dùng ID đang online (theo cài đặt tham chiếu `brute`), mảnh trong `[1, m]`. Mọi cài đặt phải cho output giống hệt.

In [15]:
def random_session(rng, n_ops, m):
    ref = FileSharingBrute(m)
    cmds, args = ["FileSharing"], [[m]]
    for _ in range(n_ops):
        online = list(ref.users)
        op = rng.choice(["join", "join", "leave", "request", "request", "request"])
        if op != "join" and not online:
            op = "join"
        if op == "join":
            a = [rng.sample(range(1, m + 1), rng.randint(0, min(3, m)))]
            ref.join(*a)
        elif op == "leave":
            a = [rng.choice(online)]
            ref.leave(*a)
        else:
            a = [rng.choice(online), rng.randint(1, m)]
            ref.request(*a)
        cmds.append(op); args.append(a)
    return cmds, args

rng = random.Random(1500)
for _ in range(600):
    m = rng.randint(1, 6)
    cmds, args = random_session(rng, rng.randint(1, 40), m)
    want = run(FileSharingBrute, cmds, args)
    for name, cls in IMPLS.items():
        assert run(cls, cmds, args) == want, name
print("600 chuỗi lệnh ngẫu nhiên: mọi cài đặt cho kết quả giống hệt")

600 chuỗi lệnh ngẫu nhiên: mọi cài đặt cho kết quả giống hệt


### 12.3 Kiểm tra invariant nội bộ của bản nộp

Sau mỗi lệnh: (a) `owners` và `user_chunks` khớp nhau (I2); (b) mọi ID trong heap đều `≤ cur` và không đang online; (c) ID online ∪ heap = `{1..cur}` (I1).

In [16]:
def check_invariants(fs):
    pairs1 = {(u, c) for u, chs in fs.user_chunks.items() for c in chs}
    pairs2 = {(u, c) for c, us in fs.owners.items() for u in us}
    assert pairs1 == pairs2, "hai chỉ mục lệch nhau"
    heap_ids = set(fs.reused)
    assert len(heap_ids) == len(fs.reused), "heap có ID trùng"
    assert not (heap_ids & set(fs.user_chunks)), "ID vừa online vừa trong heap"
    assert heap_ids | set(fs.user_chunks) == set(range(1, fs.cur + 1)), "mất ID"

for _ in range(300):
    m = rng.randint(1, 5)
    cmds, args = random_session(rng, 30, m)
    fs = FileSharing(*args[0])
    for c, a in zip(cmds[1:], args[1:]):
        getattr(fs, c)(*a)
        check_invariants(fs)
print("300 phiên: invariant I1, I2 luôn đúng")

300 phiên: invariant I1, I2 luôn đúng


### 12.4 Hiệu năng: `10^4` lời gọi

In [17]:
def stress(cls, rng_seed):
    r = random.Random(rng_seed)
    fs = cls(10**5)
    online = []
    t0 = time.perf_counter()
    for i in range(10**4):
        if i < 5000 or not online or r.random() < 0.2:
            online.append(fs.join(r.sample(range(1, 10**5 + 1), 100)))
        elif r.random() < 0.3:
            u = online.pop(r.randrange(len(online))); fs.leave(u)
        else:
            fs.request(r.choice(online), r.randint(1, 10**5))
    return time.perf_counter() - t0

for name in ["heap(doocs)", "index", "submit"]:
    print(f"{name:12}: {stress(IMPLS[name], 7):.3f}s")

heap(doocs) : 1.813s


index       : 0.593s


submit      : 0.575s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cấp ID bằng "số người hiện tại + 1"

Sau khi có người rời đi, `len(users) + 1` có thể trùng với ID của người đang online.

In [18]:
class Wrong1(FileSharingBrute):
    def join(self, ownedChunks):
        uid = len(self.users) + 1             # SAI
        self.users[uid] = set(ownedChunks)    # ghi đè người đang online!
        return uid

cmds_w = ["FileSharing", "join", "join", "leave", "join", "request"]
args_w = [[3], [[1]], [[2]], [1], [[3]], [2, 2]]
print("sai :", run(Wrong1, cmds_w, args_w))
print("đúng:", run(FileSharing, cmds_w, args_w))

sai : [None, 1, 2, None, 2, []]
đúng: [None, 1, 2, None, 1, [2]]


Bản sai cấp ID **2** lần nữa (người 2 vẫn online) và ghi đè mảnh của họ ⇒ `request(2, 2)` trả `[]` thay vì `[2]`. **Sửa:** heap + bộ đếm.

### Lỗi 2 — Quên cho người yêu cầu nhận mảnh

Đặc tả: danh sách khác rỗng ⇒ người yêu cầu **sở hữu** mảnh đó từ nay.

In [19]:
class Wrong2(FileSharingIndex):
    def request(self, userID, chunkID):
        return sorted(self.owners[chunkID])   # SAI: không cập nhật quyền sở hữu

print("sai :", run(Wrong2, EX_CMDS, EX_ARGS))
print("đúng:", run(FileSharing, EX_CMDS, EX_ARGS))
cmds_w2 = ["FileSharing", "join", "join", "request", "leave", "request"]
args_w2 = [[2], [[2]], [[]], [2, 2], [1], [2, 2]]
print("sai :", run(Wrong2, cmds_w2, args_w2), "| đúng:", run(FileSharing, cmds_w2, args_w2))

sai : [None, 1, 2, 3, [2], [1, 2], None, [], None, 1]
đúng: [None, 1, 2, 3, [2], [1, 2], None, [], None, 1]
sai : [None, 1, 2, [1], None, []] | đúng: [None, 1, 2, [1], None, [2]]


Ở ví dụ đề, bản sai vẫn cho output trùng (không lệnh nào phụ thuộc vào mảnh vừa nhận) — đây là loại lỗi chỉ lộ ra với test tự dựng. Ở chuỗi thứ hai: user 2 xin mảnh 2 từ user 1, rồi user 1 rời đi. Đúng ra user 2 vẫn có mảnh 2 ⇒ `[2]`; bản sai trả `[]`.

### Lỗi 3 — Cho nhận mảnh cả khi danh sách rỗng

Ngược với Lỗi 2: người yêu cầu "nhận" mảnh từ hư không.

In [20]:
class Wrong3(FileSharingIndex):
    def request(self, userID, chunkID):
        res = sorted(self.owners[chunkID])
        self.owners[chunkID].add(userID)      # SAI: phải kiểm tra res khác rỗng
        self.user_chunks[userID].add(chunkID)
        return res

cmds_w3 = ["FileSharing", "join", "join", "request", "request"]
args_w3 = [[3], [[]], [[]], [1, 3], [2, 3]]
print("sai :", run(Wrong3, cmds_w3, args_w3))
print("đúng:", run(FileSharing, cmds_w3, args_w3))

sai : [None, 1, 2, [], [1]]
đúng: [None, 1, 2, [], []]


### Lỗi 4 — Quên `sorted()`

Duyệt `dict` theo thứ tự chèn: ID 1 được cấp lại **sau** ID 2 ⇒ nằm sau trong `dict`. Với chỉ mục `set`, thứ tự cũng không đảm bảo.

In [21]:
class Wrong4(FileSharingHeap):
    def request(self, userID, chunkID):
        res = [k for k, v in self.user_chunks.items() if chunkID in v]   # SAI: thiếu sorted
        if res:
            self.user_chunks[userID].add(chunkID)
        return res

class Wrong4b(FileSharingIndex):
    def request(self, userID, chunkID):
        res = list(self.owners[chunkID])                                  # SAI: thiếu sorted
        if res:
            self.owners[chunkID].add(userID); self.user_chunks[userID].add(chunkID)
        return res

cmds_w4 = ["FileSharing", "join", "join", "leave", "join", "request"]
args_w4 = [[2], [[1]], [[1]], [1], [[1]], [2, 1]]
print("sai (dict) :", run(Wrong4, cmds_w4, args_w4), "| đúng:", run(FileSharing, cmds_w4, args_w4))
cmds_w4b = ["FileSharing"] + ["join"] * 9 + ["request", "request"]
args_w4b = [[5]] + [[[]]] * 8 + [[[5]]] + [[1, 5], [2, 5]]
print("sai (set)  :", run(Wrong4b, cmds_w4b, args_w4b)[-1], "| đúng:", run(FileSharing, cmds_w4b, args_w4b)[-1])

sai (dict) : [None, 1, 2, None, 1, [2, 1]] | đúng: [None, 1, 2, None, 1, [1, 2]]
sai (set)  : [9, 1] | đúng: [1, 9]


### Lỗi 5 — `leave` không dọn chỉ mục ngược

Chỉ xoá `user_chunks[uid]` mà quên `owners[c]` ⇒ người đã rời vẫn xuất hiện trong kết quả `request` ("người dùng ma"); tệ hơn, nếu ID đó được cấp lại cho người mới, người mới bị coi là đang có mảnh của người cũ.

In [22]:
class Wrong5(FileSharingIndex):
    def leave(self, userID):
        self.user_chunks.pop(userID)          # SAI: không xoá khỏi owners
        heapq.heappush(self.reused, userID)

cmds_w5 = ["FileSharing", "join", "join", "leave", "request", "join", "request"]
args_w5 = [[3], [[3]], [[]], [1], [2, 3], [[]], [2, 3]]
print("sai :", run(Wrong5, cmds_w5, args_w5))
print("đúng:", run(FileSharing, cmds_w5, args_w5))

sai : [None, 1, 2, None, [1], 1, [1, 2]]
đúng: [None, 1, 2, None, [], 1, []]


**Sửa:** `leave` duyệt mọi mảnh của người dùng và `discard` khỏi `owners`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Cấp **số nhỏ nhất chưa dùng**, có trả lại" ⇒ **bộ đếm + min-heap** (giống LC 2336 Smallest Number in Infinite Set, LC 1845 Seat Reservation Manager).
- Truy vấn "ai có X?" thường xuyên ⇒ **chỉ mục ngược** X → tập người; nhớ cập nhật **hai phía**.
- Kết quả yêu cầu "tăng dần" ⇒ `sorted()`; đừng tin thứ tự của `set`/`dict`.
- Viết **bộ chạy lệnh** + cài đặt brute force trước, rồi đối chiếu ngẫu nhiên — cách nhanh nhất để bắt lỗi trạng thái trong bài thiết kế.
- Viết hàm kiểm tra invariant (hai chỉ mục khớp, ID không mất/không trùng).

## 15. Biến thể của bài

Các câu follow-up của đề:

1. **Định danh theo IP, ngắt kết nối rồi vào lại cùng IP:** ID không còn do hệ thống cấp; dùng `dict` IP → tập mảnh; khi "vào lại" có thể khôi phục mảnh cũ (nếu muốn giữ) thay vì cấp ID mới. Heap ID không cần nữa.
2. **Join/leave liên tục, không request:** Cách 2/3 đều `O(log U)` cho mỗi lần (Cách 3 thêm `O(c)` để cập nhật chỉ mục) ⇒ vẫn hiệu quả. Cách 1 (quét ID) thì không.
3. **Mọi người join, xin hết các mảnh rồi leave:** mỗi người gom tới `m` mảnh ⇒ `leave` ở Cách 3 tốn `O(m)`; chỉ mục ngược phình to `O(U·m)`. Có thể lưu theo **khoảng mảnh** (interval) hoặc bitset để nén.
4. **Nhiều file, file `i` có `m[i]` mảnh:** đổi khoá mảnh thành cặp `(fileID, chunkID)` (Bài 5).
5. **Bài cùng họ:** LC 2336, LC 1845, LC 379 Design Phone Directory (cấp số nhỏ nhất / bất kỳ còn trống).

In [23]:
# Minh hoạ follow-up 1: định danh theo IP (người dùng giữ lại mảnh khi vào lại)
class FileSharingIP:
    def __init__(self, m):
        self.m = m
        self.online = set()
        self.chunks_of = defaultdict(set)        # ip -> mảnh (giữ cả khi offline)
    def connect(self, ip, ownedChunks=()):
        self.online.add(ip); self.chunks_of[ip] |= set(ownedChunks)
    def disconnect(self, ip):
        self.online.discard(ip)
    def request(self, ip, chunkID):
        res = sorted(u for u in self.online if chunkID in self.chunks_of[u])
        if res:
            self.chunks_of[ip].add(chunkID)
        return res

fip = FileSharingIP(3)
fip.connect("10.0.0.2", [1]); fip.connect("10.0.0.1", [])
print(fip.request("10.0.0.1", 1)); fip.disconnect("10.0.0.2"); fip.disconnect("10.0.0.1")
fip.connect("10.0.0.1"); fip.connect("10.0.0.3")
print(fip.request("10.0.0.3", 1), "(10.0.0.1 vào lại vẫn còn mảnh 1)")

['10.0.0.2']
['10.0.0.1'] (10.0.0.1 vào lại vẫn còn mảnh 1)


## 16. So sánh các cách

| Cách | Ý tưởng | join | request | Khi nào dùng |
|---|---|---|---|---|
| 1. Quét | Thử ID từ 1; duyệt mọi người | `O(U)` | `O(U log U)` | Tham chiếu để test |
| 2. Heap (doocs) | Bộ đếm + heap ID trả lại; duyệt người khi request | `O(log U + c)` | `O(U log U)` | Gọn, đủ với `10^4` lời gọi |
| 3. Heap + chỉ mục ngược | Thêm `owners[chunk]` | `O(log U + c)` | `O(k log k)` | Request nhiều, nhiều người online |
| 4. Set + `min` | ID trống trong `set`, lấy `min` | `O(|free| + c)` | `O(k log k)` | Minh hoạ vai trò của heap |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
Chạy chuỗi `["FileSharing","join","join","join","leave","join","request"]` với `[[2],[[1]],[[]],[[2]],[1],[[]],[1,1]]` cho output gì?

### Bài 2 (tính tay)
Với chuỗi: 4 lần `join([])`, rồi `leave(3)`, `leave(1)`, `leave(4)`, rồi 3 lần `join([])`: ghi `cur`, heap sau mỗi lệnh và 3 ID được cấp cuối cùng.

### Bài 3 (điền TODO)
Hoàn thiện lớp `IdPool` với `acquire() -> int` (ID nhỏ nhất còn trống) và `release(x)`.

### Bài 4 (tự viết lại)
Viết lại `IdPool` bằng danh sách `free` **luôn được sắp xếp** (dùng `bisect.insort`) thay cho heap: `IdPoolSorted`.

### Bài 5 (mini challenge)
Viết `FileSharingMulti(ms)` cho **nhiều file** (`ms[i]` = số mảnh của file `i`, file đánh số từ 0): `join(owned)` với `owned` là list cặp `(fileID, chunkID)`, `leave(uid)`, `request(uid, fileID, chunkID)`.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
class IdPool:
    def __init__(self):
        # TODO: self.cur = 0; self.heap = []
        pass
    def acquire(self):
        # TODO: lấy đỉnh heap nếu có, ngược lại cur += 1
        pass
    def release(self, x):
        # TODO: heappush
        pass

# Bài 4 — TODO
class IdPoolSorted:
    def __init__(self):
        # TODO: self.cur = 0; self.free = []  (luôn tăng dần)
        pass
    def acquire(self):
        # TODO: self.free.pop(0) nếu có
        pass
    def release(self, x):
        # TODO: bisect.insort(self.free, x)
        pass

# Bài 5 — TODO
class FileSharingMulti:
    def __init__(self, ms):
        # TODO: như Cách 3 nhưng khoá mảnh là (fileID, chunkID)
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba lần join ⇒ 1, 2, 3. (2) `leave(1)` ⇒ heap `[1]`; `join([])` ⇒ **1** (người mới, không có mảnh). (3) `request(1, 1)`: người giữ mảnh 1 cũ đã rời ⇒ `[]`.

**Bài 2.** (1) Sau 4 join: `cur = 4`, heap rỗng. (2) Sau ba lần leave: heap chứa `{1, 3, 4}`. (3) Ba lần join lấy `1, 3, 4`; `cur` vẫn là 4.

**Bài 3.** (1) Hai thuộc tính `cur`, `heap`. (2) `acquire`: `heappop` nếu heap khác rỗng. (3) Kiểm chứng bằng cách so với "ID nhỏ nhất không thuộc tập đang dùng".

**Bài 4.** (1) `free` tăng dần ⇒ nhỏ nhất ở `free[0]`. (2) `pop(0)` là `O(len)` nhưng đủ dùng. (3) `insort` giữ thứ tự khi trả lại.

**Bài 5.** (1) `owners[(f, c)]`, `user_chunks[uid]` là set các cặp. (2) Mọi thao tác như Cách 3. (3) Kiểm tra: với một file duy nhất phải cho kết quả giống `FileSharing`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
cmds_b1 = ["FileSharing", "join", "join", "join", "leave", "join", "request"]
args_b1 = [[2], [[1]], [[]], [[2]], [1], [[]], [1, 1]]
out_b1 = run(FileSharing, cmds_b1, args_b1)
print(out_b1)
assert out_b1 == [None, 1, 2, 3, None, 1, []]

# Bài 2
fs = FileSharing(1)
ids = [fs.join([]) for _ in range(4)]
for u in [3, 1, 4]:
    fs.leave(u)
print("heap:", sorted(fs.reused), "cur:", fs.cur)
last3 = [fs.join([]) for _ in range(3)]
print("3 ID cuối:", last3, "cur:", fs.cur)
assert ids == [1, 2, 3, 4] and last3 == [1, 3, 4] and fs.cur == 4

[None, 1, 2, 3, None, 1, []]
heap: [1, 3, 4] cur: 4
3 ID cuối: [1, 3, 4] cur: 4


In [26]:
# Bài 3, Bài 4
import bisect
class IdPoolSol:
    def __init__(self):
        self.cur = 0; self.heap = []
    def acquire(self):
        if self.heap:
            return heapq.heappop(self.heap)
        self.cur += 1
        return self.cur
    def release(self, x):
        heapq.heappush(self.heap, x)

class IdPoolSortedSol:
    def __init__(self):
        self.cur = 0; self.free = []
    def acquire(self):
        if self.free:
            return self.free.pop(0)
        self.cur += 1
        return self.cur
    def release(self, x):
        bisect.insort(self.free, x)

rng = random.Random(19)
for _ in range(300):
    p1, p2, used = IdPoolSol(), IdPoolSortedSol(), set()
    for _ in range(60):
        if used and rng.random() < 0.4:
            x = rng.choice(sorted(used)); used.remove(x)
            p1.release(x); p2.release(x)
        else:
            want = next(i for i in range(1, 1000) if i not in used)
            a, b = p1.acquire(), p2.acquire()
            assert a == b == want
            used.add(a)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [27]:
# Bài 5
class FileSharingMultiSol:
    def __init__(self, ms):
        self.ms = ms
        self.cur = 0
        self.reused = []
        self.user_chunks = {}
        self.owners = defaultdict(set)
    def join(self, owned):
        if self.reused:
            uid = heapq.heappop(self.reused)
        else:
            self.cur += 1; uid = self.cur
        self.user_chunks[uid] = {tuple(p) for p in owned}
        for p in self.user_chunks[uid]:
            self.owners[p].add(uid)
        return uid
    def leave(self, uid):
        for p in self.user_chunks.pop(uid):
            self.owners[p].discard(uid)
        heapq.heappush(self.reused, uid)
    def request(self, uid, fileID, chunkID):
        key = (fileID, chunkID)
        res = sorted(self.owners[key])
        if res:
            self.owners[key].add(uid); self.user_chunks[uid].add(key)
        return res

mf = FileSharingMultiSol([3, 2])
print(mf.join([(0, 1), (1, 2)]), mf.join([(0, 1)]), mf.request(2, 1, 2), mf.request(1, 0, 3))
# Với một file duy nhất (fileID = 0) phải giống FileSharing
for _ in range(300):
    m = rng.randint(1, 5)
    cmds, args = random_session(rng, 30, m)
    want = run(FileSharing, cmds, args)
    obj = FileSharingMultiSol([m]); got = [None]
    for c, a in zip(cmds[1:], args[1:]):
        if c == "join":
            got.append(obj.join([(0, x) for x in a[0]]))
        elif c == "leave":
            got.append(obj.leave(a[0]))
        else:
            got.append(obj.request(a[0], 0, a[1]))
    assert got == want
print("Đáp án Bài 1–5: mọi assert đều qua")

1 2 [1] []
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "cấp ID nhỏ nhất, có tái sử dụng":**

```python
self.cur, self.reused = 0, []          # ID > cur chưa từng dùng; heap ID trả lại
def acquire(self):
    if self.reused:
        return heapq.heappop(self.reused)
    self.cur += 1
    return self.cur
def release(self, x):
    heapq.heappush(self.reused, x)
```

**Template chỉ mục hai chiều:**

```python
user_chunks[uid] = set(chunks); for c in chunks: owners[c].add(uid)      # join
for c in user_chunks.pop(uid): owners[c].discard(uid)                     # leave
res = sorted(owners[c]); if res: owners[c].add(uid); user_chunks[uid].add(c)   # request
```

**Checklist trước khi submit:**
- [ ] ID mới = nhỏ nhất còn trống (heap + bộ đếm), không phải `len(users) + 1`.
- [ ] `request` trả danh sách **đã sắp xếp**.
- [ ] Người yêu cầu nhận mảnh **chỉ khi** danh sách khác rỗng.
- [ ] `leave` dọn **cả hai** chỉ mục; ID được đẩy vào heap.
- [ ] Đã test bằng bộ chạy lệnh giả lập + đối chiếu cài đặt brute force.
- [ ] Đúng tên lớp/method: `FileSharing(m)`, `join(ownedChunks)`, `leave(userID)`, `request(userID, chunkID)`.